In [1]:
!pip install zss -q
!pip install antlr4-python3-runtime==4.11 -q

  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 5.9 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.0 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.11.0 which is incompatible.


In [2]:
from kaggle_secrets import UserSecretsClient
secret_label = "GITHUB_TOKEN"
secret_value = UserSecretsClient().get_secret(secret_label)

In [3]:
import os

token = secret_value

repo_url = f"https://{token}@github.com/smiling-demon/llm-symbolic-ode-reasoning-dev.git"
repo_dir = "llm-symbolic-ode-reasoning-dev"

if not os.path.exists(repo_dir):
    !git clone {repo_url}

os.chdir(repo_dir)

!pwd

Cloning into 'llm-symbolic-ode-reasoning-dev'...
remote: Enumerating objects: 135, done.
remote: Counting objects: 100% (135/135), done.
remote: Compressing objects: 100% (100/100), done.
remote: Total 135 (delta 54), reused 112 (delta 31), pack-reused 0 (from 0)
Receiving objects: 100% (135/135), 1.53 MiB | 19.08 MiB/s, done.
Resolving deltas: 100% (54/54), done.
/kaggle/working/llm-symbolic-ode-reasoning-dev


In [4]:
import pandas as pd
from sentence_transformers import SentenceTransformer

from llm import LLM
from methods import train_reasoning_bank, rsa_bank
from metrics import evaluate_predictions

In [5]:
df_train = pd.read_excel("/kaggle/input/datasets/dmitrylessy/ode-basic-dataset/data/train.xlsx")
df_test = pd.read_excel("/kaggle/input/datasets/dmitrylessy/ode-basic-dataset/data/test.xlsx")


equations_train = df_train['equation'].tolist()[:20]
solutions_train = df_train['solution'].tolist()[:20]

equations_test = df_test['equation'].tolist()[:20]
solutions_test = df_test['solution'].tolist()[:20]

In [6]:
llm = LLM("Qwen/Qwen2.5-3B-Instruct")

embedding_model = SentenceTransformer("BAAI/bge-base-en-v1.5", device="cuda")

config.json:   0%|          | 0.00/661 [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

...

tokenizer_config.json:   0%|          | 0.00/366 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/125 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
%%time

batch_size = 10

bank = train_reasoning_bank(llm,
                            embedding_model,
                            equations_train,
                            solutions_train,
                            batch_size=batch_size,
                            max_new_tokens=1024)

CPU times: user 3min 50s, sys: 1.53 s, total: 3min 52s
Wall time: 3min 52s


In [8]:
%%time

batch_size = 5
predictions = []

for i in range(0, len(equations_test), batch_size):
    predictions += rsa_bank(llm,
                       bank,
                       embedding_model,
                       equations_test[i: i + batch_size],
                       max_new_tokens=1024)

CPU times: user 27min 27s, sys: 1min 10s, total: 28min 38s
Wall time: 28min 36s


In [9]:
import json

with open("/kaggle/working/predictions.json", "w", encoding="utf-8") as f:
    json.dump(predictions, f, ensure_ascii=False, indent=2)

In [10]:
evaluation = evaluate_predictions(predictions, solutions_test)
for key in evaluation:
    evaluation[key] = [evaluation[key]]
pd.DataFrame.from_dict(evaluation)

,bleu,ast_error_size,ast_tree_similarity,exact_match
0,0.721803,0.681622,0.504527,0.3
